# Baseball Win Probability — Data Preprocessing

Pipeline:
1. Parse gamelogs → `data/games.csv`
2. Map player IDs (Retrosheet → FanGraphs) → `data/player_id_map.csv`
3. Fetch & cache per-date player stats → `data/stats_cache/`
4. Build feature matrix → `data/train_features.parquet`, `data/test_features.parquet`

## Step 1 — Parse Gamelogs

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import pybaseball
import requests
import io
from datetime import timedelta, date as date_type

/Users/kaitlynoikle/cs-projects/ml-final-project/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [4]:
DATA_DIR = Path('data')
GAMELOG_FILES = [DATA_DIR / 'gl2023.txt', DATA_DIR / 'gl2024.txt', DATA_DIR / 'gl2025.txt']

In [7]:
# Retrosheet gamelog column indices (0-indexed)
# Reference: data/gamelog_col_description.txt
COL_DATE       = 0
COL_AWAY_TEAM  = 3
COL_HOME_TEAM  = 6
COL_AWAY_SCORE = 9
COL_HOME_SCORE = 10
COL_AWAY_SP_ID = 101  # visiting starting pitcher Retrosheet ID
COL_HOME_SP_ID = 103  # home starting pitcher Retrosheet ID

# Lineup ID columns: every 3rd column (ID, name, position)
AWAY_LINEUP_COLS = [105 + i * 3 for i in range(9)]
HOME_LINEUP_COLS = [132 + i * 3 for i in range(9)]

def parse_gamelogs(files):
    frames = [pd.read_csv(f, header=None, dtype=str) for f in files]
    raw = pd.concat(frames, ignore_index=True)

    games = pd.DataFrame()
    games['date']       = pd.to_datetime(raw[COL_DATE], format='%Y%m%d')
    games['away_team']  = raw[COL_AWAY_TEAM]
    games['home_team']  = raw[COL_HOME_TEAM]
    games['away_score'] = pd.to_numeric(raw[COL_AWAY_SCORE])
    games['home_score'] = pd.to_numeric(raw[COL_HOME_SCORE])
    games['home_win']   = (games['home_score'] > games['away_score']).astype(int)
    games['away_sp_id'] = raw[COL_AWAY_SP_ID].str.strip('"')
    games['home_sp_id'] = raw[COL_HOME_SP_ID].str.strip('"')

    for i, col in enumerate(AWAY_LINEUP_COLS):
        games[f'away_bat{i+1}_id'] = raw[col].str.strip('"')
    for i, col in enumerate(HOME_LINEUP_COLS):
        games[f'home_bat{i+1}_id'] = raw[col].str.strip('"')

    # Drop ties (rare suspended games)
    games = games[games['away_score'] != games['home_score']].reset_index(drop=True)
    return games

games = parse_gamelogs(GAMELOG_FILES)
print(f'Parsed {len(games)} games')
games.head()

Parsed 7289 games


,date,away_team,home_team,away_score,home_score,home_win,away_sp_id,home_sp_id,away_bat1_id,away_bat2_id,...,away_bat9_id,home_bat1_id,home_bat2_id,home_bat3_id,home_bat4_id,home_bat5_id,home_bat6_id,home_bat7_id,home_bat8_id,home_bat9_id
0,2023-03-30,MIL,CHN,0,4,1,burnc002,strom001,yelic001,winkj002,...,turab002,hoern001,swand001,happi001,bellc002,manct001,gomey001,hosme001,wisdp001,mastm001
1,2023-03-30,PIT,CIN,5,4,0,kellm003,greeh001,cruzo001,reynb001,...,hedga001,indij001,friet001,fralj001,stept001,voslj001,myerw001,stees001,bensw001,garcj007
2,2023-03-30,ARI,LAN,2,8,1,gallz001,uriaj001,lewik001,martk001,...,mccaj003,bettm001,freef001,smitw003,muncm001,martj006,perad001,vargm001,outmj002,rojam002
3,2023-03-30,NYN,MIA,5,3,0,schem001,alcas001,nimmb001,marts002,...,narvo001,arral001,seguj002,coopg002,chisj001,solej001,garca003,delab001,stalj001,wendj002
4,2023-03-30,COL,SDN,7,2,0,marqg001,snelb001,dazay001,bryak001,...,tovae001,grist001,sotoj001,machm001,bogax001,cronj001,carpm002,nolaa002,kim-h002,dahld001


In [ ]:
games.to_csv(DATA_DIR / 'games.csv', index=False)
print('Saved data/games.csv')

## Step 2 — Player ID Mapping (Retrosheet → FanGraphs)

In [8]:
# Collect all unique Retrosheet player IDs appearing in our games
id_cols = (['away_sp_id', 'home_sp_id'] +
           [f'away_bat{i}_id' for i in range(1, 10)] +
           [f'home_bat{i}_id' for i in range(1, 10)])

all_retro_ids = (pd.concat([games[c] for c in id_cols])
                   .dropna()
                   .unique())
all_retro_ids = [rid for rid in all_retro_ids if rid.strip()]
print(f'{len(all_retro_ids)} unique Retrosheet player IDs found')

1519 unique Retrosheet player IDs found


In [13]:
# Fetch the Chadwick Bureau register directly — now split into 16 files (people-0.csv .. people-f.csv)
CHADWICK_BASE = "https://raw.githubusercontent.com/chadwickbureau/register/master/data/people-{}.csv"

chunks = []
for suffix in [*'0123456789', 'a', 'b', 'c', 'd', 'e', 'f']:
    url = CHADWICK_BASE.format(suffix)
    resp = requests.get(url)
    resp.raise_for_status()
    chunks.append(pd.read_csv(io.StringIO(resp.content.decode('utf-8')), low_memory=False))

raw_register = pd.concat(chunks, ignore_index=True)
print("Columns:", raw_register.columns.tolist())
print("Shape:", raw_register.shape)

Columns: ['key_person', 'key_uuid', 'key_mlbam', 'key_retro', 'key_bbref', 'key_bbref_minors', 'key_fangraphs', 'key_npb', 'key_sr_nfl', 'key_sr_nba', 'key_sr_nhl', 'key_wikidata', 'name_last', 'name_first', 'name_given', 'name_suffix', 'name_matrilineal', 'name_nick', 'birth_year', 'birth_month', 'birth_day', 'death_year', 'death_month', 'death_day', 'pro_played_first', 'pro_played_last', 'mlb_played_first', 'mlb_played_last', 'col_played_first', 'col_played_last', 'pro_managed_first', 'pro_managed_last', 'mlb_managed_first', 'mlb_managed_last', 'col_managed_first', 'col_managed_last', 'pro_umpired_first', 'pro_umpired_last', 'mlb_umpired_first', 'mlb_umpired_last']
Shape: (510627, 40)


In [14]:
# Build retro_id → mlbam_id mapping filtered to our player set
subset = raw_register[raw_register['key_retro'].isin(all_retro_ids)].copy()

mapping = (
    subset[['key_retro', 'key_mlbam', 'name_first', 'name_last']]
    .rename(columns={'key_retro': 'retro_id', 'key_mlbam': 'mlbam_id'})
)
mapping['name'] = mapping['name_first'].str.strip() + ' ' + mapping['name_last'].str.strip()
mapping = (mapping
    .drop(columns=['name_first', 'name_last'])
    .drop_duplicates(subset='retro_id')
    .reset_index(drop=True)
)

not_in_register = [rid for rid in all_retro_ids if rid not in set(mapping['retro_id'])]
missing_mlbam   = mapping[mapping['mlbam_id'].isna()]['retro_id'].tolist()
has_mlbam       = mapping[mapping['mlbam_id'].notna()].copy()
has_mlbam['mlbam_id'] = has_mlbam['mlbam_id'].astype(int)

print(f'Total players:            {len(all_retro_ids)}')
print(f'Not in register:          {len(not_in_register)}')
print(f'In register, no MLBAM ID: {len(missing_mlbam)}')
print(f'Full mapping (usable):    {len(has_mlbam)}')

if missing_mlbam:
    print('\nMissing MLBAM ID sample:', missing_mlbam[:10])

has_mlbam.head()

Total players:            1519
Not in register:          0
In register, no MLBAM ID: 0
Full mapping (usable):    1519


,retro_id,mlbam_id,name
0,millo002,680911,Owen Miller
1,edmat001,669242,Tommy Edman
2,castd004,660636,Diego Castillo
3,cessl001,570666,Luis Cessa
4,zastr001,642239,Rob Zastryzny


In [16]:
# Filter games to only those where every lineup player has an MLBAM ID
players_with_mlbam = set(has_mlbam['retro_id'])

def all_players_have_mlbam(row):
    for col in id_cols:
        pid = row.get(col)
        if pd.notna(pid) and pid.strip() and pid not in players_with_mlbam:
            return False
    return True

mask = games.apply(all_players_have_mlbam, axis=1)
games_clean = games[mask].reset_index(drop=True)

print(f'Games before filter: {len(games)}')
print(f'Games after filter:  {len(games_clean)}  ({len(games) - len(games_clean)} dropped)')
print(f'Date range: {games_clean["date"].min()} → {games_clean["date"].max()}')

Games before filter: 7289
Games after filter:  7289  (0 dropped)
Date range: 2023-03-30 00:00:00 → 2025-09-28 00:00:00


In [15]:
mapping.to_csv(DATA_DIR / 'player_id_map.csv', index=False)
print('Saved data/player_id_map.csv')

Saved data/player_id_map.csv


## Step 3 — Fetch & Cache Player Stats

**`TIME_WINDOW` options:**
- `"season_to_date"` — stats from opening day through game_date − 1 *(default, no leakage)*
- `"trailing_N"` — last N days (set `WINDOW_DAYS`); captures recent form
- `"full_prior_season"` — full previous calendar year

Stats are cached to `data/stats_cache/{window}/` so re-running is instant.

In [17]:
STATS_CACHE_DIR = DATA_DIR / 'stats_cache'

# Approximate MLB opening day by season year
OPENING_DAYS = {
    2022: date_type(2022, 4, 7),
    2023: date_type(2023, 3, 30),
    2024: date_type(2024, 3, 20),
    2025: date_type(2025, 3, 27),
}


def get_stat_window(game_date, time_window='season_to_date', window_days=30):
    """Return (start_date, end_date, cache_label) for a game date and window type."""
    d = game_date if isinstance(game_date, date_type) else game_date.date()
    end = d - timedelta(days=1)

    if time_window == 'season_to_date':
        start = OPENING_DAYS[d.year]
        label = 'season_to_date'
    elif time_window == 'trailing_N':
        start = d - timedelta(days=window_days)
        label = f'trailing_{window_days}'
    elif time_window == 'full_prior_season':
        start = OPENING_DAYS.get(d.year - 1, date_type(d.year - 1, 3, 28))
        end   = date_type(d.year - 1, 11, 1)
        label = f'prior_season_{d.year - 1}'
    elif time_window == 'full_current_season':
        start = OPENING_DAYS[d.year]
        end   = date_type(d.year, 11, 1)
        label = f'full_current_season_{d.year}'
    else:
        raise ValueError(f'Unknown time_window: {time_window}')

    return start, end, label


def fetch_batting_stats_cached(start, end, cache_label):
    cache_dir  = STATS_CACHE_DIR / cache_label / 'batting'
    cache_dir.mkdir(parents=True, exist_ok=True)
    cache_file = cache_dir / f'{start}_{end}.csv'

    if cache_file.exists():
        return pd.read_csv(cache_file)

    df = pybaseball.batting_stats_range(str(start), str(end))
    df.to_csv(cache_file, index=False)
    return df


def fetch_pitching_stats_cached(start, end, cache_label):
    cache_dir  = STATS_CACHE_DIR / cache_label / 'pitching'
    cache_dir.mkdir(parents=True, exist_ok=True)
    cache_file = cache_dir / f'{start}_{end}.csv'

    if cache_file.exists():
        return pd.read_csv(cache_file)

    df = pybaseball.pitching_stats_range(str(start), str(end))
    df.to_csv(cache_file, index=False)
    return df


print('Stat fetch functions defined.')

Stat fetch functions defined.


In [15]:
# ── Configuration ──────────────────────────────────────────────
TIME_WINDOW = 'full_current_season'   # 'season_to_date' | 'trailing_N' | 'full_prior_season' | 'full_current_season'
WINDOW_DAYS = 30                 # only used when TIME_WINDOW == 'trailing_N'
# ───────────────────────────────────────────────────────────────

all_dates = sorted(games['date'].dt.date.unique())
print(f'Fetching stats for {len(all_dates)} unique game dates (window={TIME_WINDOW})')
print('First run takes a few minutes; subsequent runs use cache.')

Fetching stats for 551 unique game dates (window=full_current_season)
First run takes a few minutes; subsequent runs use cache.


In [18]:
batting_stats_by_date  = {}
pitching_stats_by_date = {}

for i, game_date in enumerate(all_dates):
    start, end, label = get_stat_window(game_date, TIME_WINDOW, WINDOW_DAYS)

    if start >= end:
        # Opening day or before — no prior stats available yet
        batting_stats_by_date[game_date]  = pd.DataFrame()
        pitching_stats_by_date[game_date] = pd.DataFrame()
        continue

    batting_stats_by_date[game_date]  = fetch_batting_stats_cached(start, end, label)
    pitching_stats_by_date[game_date] = fetch_pitching_stats_cached(start, end, label)

    if (i + 1) % 50 == 0:
        print(f'  {i + 1}/{len(all_dates)} dates done')

print('Done.')

  50/551 dates done
  100/551 dates done
  150/551 dates done
  200/551 dates done
  250/551 dates done
  300/551 dates done
  350/551 dates done
  400/551 dates done
  450/551 dates done
  500/551 dates done
  550/551 dates done
Done.


In [19]:
# Inspect available stat columns — used to confirm column names before Step 4
sample_date = all_dates[60]
print('Batting columns:')
print(batting_stats_by_date[sample_date].columns.tolist())
print()
print('Pitching columns:')
print(pitching_stats_by_date[sample_date].columns.tolist())

Batting columns:
['Name', 'Age', '#days', 'Lev', 'Tm', 'G', 'PA', 'AB', 'R', 'H', '2B', '3B', 'HR', 'RBI', 'BB', 'IBB', 'SO', 'HBP', 'SH', 'SF', 'GDP', 'SB', 'CS', 'BA', 'OBP', 'SLG', 'OPS', 'mlbID']

Pitching columns:
['Name', 'Age', '#days', 'Lev', 'Tm', 'G', 'GS', 'W', 'L', 'SV', 'IP', 'H', 'R', 'ER', 'BB', 'SO', 'HR', 'HBP', 'ERA', 'AB', '2B', '3B', 'IBB', 'GDP', 'SF', 'SB', 'CS', 'PO', 'BF', 'Pit', 'Str', 'StL', 'StS', 'GB/FB', 'LD', 'PU', 'WHIP', 'BAbip', 'SO9', 'SO/W', 'mlbID']


## Step 4 — Build Feature Matrix

In [ ]:
# Load the ID map built in Step 2
id_map     = pd.read_csv(DATA_DIR / 'player_id_map.csv')
retro_to_mlbam = dict(zip(id_map['retro_id'], id_map['mlbam_id'].astype('Int64')))

# Stats to include per player type
# batting_stats_range returns: BA, OBP, SLG, OPS, BB, SO, PA (no K%, BB%, wRC+)
# We compute K% and BB% from raw columns
BATTING_STATS  = ['BA', 'OBP', 'SLG', 'OPS', 'K%', 'BB%']
PITCHING_STATS = ['ERA', 'WHIP', 'SO9', 'SO/W', 'IP']
# Note: K/9 → SO9, BB/9 not available, FIP not available from batting_stats_range


def lookup_batting_stats(mlbam_id, stats_df):
    """Return dict of batting stats for a MLBAM player ID; NaN for any missing values."""
    result = {col: np.nan for col in BATTING_STATS}
    if stats_df.empty or pd.isna(mlbam_id):
        return result
    row = stats_df[stats_df['mlbID'] == mlbam_id]
    if row.empty:
        return result
    r = row.iloc[0]
    result['BA']  = r.get('BA', np.nan)
    result['OBP'] = r.get('OBP', np.nan)
    result['SLG'] = r.get('SLG', np.nan)
    result['OPS'] = r.get('OPS', np.nan)
    # Compute rate stats from counting stats
    pa = r.get('PA', np.nan)
    if pa and pa > 0:
        result['K%'] = round(r.get('SO', np.nan) / pa, 2)
        result['BB%'] = round(r.get('BB', np.nan) / pa, 2)
    return result


def lookup_pitching_stats(mlbam_id, stats_df):
    """Return dict of pitching stats for a MLBAM player ID; NaN for any missing values."""
    result = {col: np.nan for col in PITCHING_STATS}
    if stats_df.empty or pd.isna(mlbam_id):
        return result
    row = stats_df[stats_df['mlbID'] == mlbam_id]
    if row.empty:
        return result
    r = row.iloc[0]
    for col in PITCHING_STATS:
        result[col] = r.get(col, np.nan)
    return result


def build_game_features(game_row, bat_stats_by_date, pitch_stats_by_date):
    """Flatten all player stats for a single game into a feature dict."""
    game_date = game_row['date'].date()
    b_df = bat_stats_by_date.get(game_date, pd.DataFrame())
    p_df = pitch_stats_by_date.get(game_date, pd.DataFrame())
    feat = {}

    for team in ('away', 'home'):
        for i in range(1, 10):
            retro_id = game_row.get(f'{team}_bat{i}_id', '')
            mlbam_id = retro_to_mlbam.get(str(retro_id), pd.NA)
            stats    = lookup_batting_stats(mlbam_id, b_df)
            for stat, val in stats.items():
                feat[f'{team}_bat{i}_{stat}'] = val

        sp_retro = game_row.get(f'{team}_sp_id', '')
        sp_mlbam = retro_to_mlbam.get(str(sp_retro), pd.NA)
        sp_stats = lookup_pitching_stats(sp_mlbam, p_df)
        for stat, val in sp_stats.items():
            feat[f'{team}_sp_{stat}'] = val

    return feat


print('Feature builder defined.')
print(f'Batting stats: {BATTING_STATS}')
print(f'Pitching stats: {PITCHING_STATS}')


Feature builder defined.
Batting stats: ['BA', 'OBP', 'SLG', 'OPS', 'K%', 'BB%']
Pitching stats: ['ERA', 'WHIP', 'SO9', 'SO/W', 'IP']


In [31]:
rows = []
for _, row in games.iterrows():
    feat = build_game_features(row, batting_stats_by_date, pitching_stats_by_date)
    feat['date']      = row['date']
    feat['home_team'] = row['home_team']
    feat['away_team'] = row['away_team']
    feat['home_win']  = row['home_win']
    rows.append(feat)

features_df = pd.DataFrame(rows)
print(f'Feature matrix shape: {features_df.shape}')

feat_cols   = [c for c in features_df.columns if c not in ('date', 'home_team', 'away_team', 'home_win')]
missing_pct = features_df[feat_cols].isna().mean().mean() * 100
print(f'Missing feature rate: {missing_pct:.1f}%')

Feature matrix shape: (7289, 122)
Missing feature rate: 0.0%


In [32]:
print('Sample features:')
for k, v in list(build_game_features(games.iloc[0], batting_stats_by_date, pitching_stats_by_date).items())[:50]:
    print(f'  {k}: {v}')

Sample features:
  away_bat1_BA: 0.281
  away_bat1_OBP: 0.374
  away_bat1_SLG: 0.45
  away_bat1_OPS: 0.824
  away_bat1_K%: 0.21962616822429906
  away_bat1_BB%: 0.12461059190031153
  away_bat2_BA: 0.196
  away_bat2_OBP: 0.317
  away_bat2_SLG: 0.244
  away_bat2_OPS: 0.561
  away_bat2_K%: 0.2613065326633166
  away_bat2_BB%: 0.1306532663316583
  away_bat3_BA: 0.221
  away_bat3_OBP: 0.314
  away_bat3_SLG: 0.411
  away_bat3_OPS: 0.724
  away_bat3_K%: 0.2585139318885449
  away_bat3_BB%: 0.11145510835913312
  away_bat4_BA: 0.215
  away_bat4_OBP: 0.291
  away_bat4_SLG: 0.376
  away_bat4_OPS: 0.667
  away_bat4_K%: 0.245014245014245
  away_bat4_BB%: 0.09971509971509972
  away_bat5_BA: 0.29
  away_bat5_OBP: 0.368
  away_bat5_SLG: 0.455
  away_bat5_OPS: 0.823
  away_bat5_K%: 0.20772946859903382
  away_bat5_BB%: 0.10305958132045089
  away_bat6_BA: 0.194
  away_bat6_OBP: 0.337
  away_bat6_SLG: 0.299
  away_bat6_OPS: 0.636
  away_bat6_K%: 0.23163841807909605
  away_bat6_BB%: 0.11864406779661017
  away

In [ ]:
# Fill missing values with column mean (league average proxy)
col_means = features_df[feat_cols].mean()
features_df[feat_cols] = features_df[feat_cols].fillna(col_means)

# Train / test split: 2023-2024 train, 2025 held out
train_df = features_df[features_df['date'].dt.year.isin([2023, 2024])].reset_index(drop=True)
test_df  = features_df[features_df['date'].dt.year == 2025].reset_index(drop=True)

print(f'Train: {len(train_df)} games | Test: {len(test_df)} games')

train_df.to_parquet(DATA_DIR / 'train_features.parquet', index=False)
test_df.to_parquet(DATA_DIR / 'test_features.parquet', index=False)
print('Saved train_features.parquet and test_features.parquet')